[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/18_montecarlo_search/notebooks/aplicaciones/04_torneo.ipynb)

# Notebook 4: Torneo de agentes en Hex

**Módulo 18 — Monte Carlo Tree Search · Notebook 04 (Aplicaciones)**

---

## ¿De qué trata este notebook?

En este notebook construimos un **torneo round-robin** entre tres agentes
que juegan Hex 5×5: aleatorio, MCTS (UCT) y alpha-beta con función de
evaluación heurística.

**Lo que vas a construir y explorar:**

1. Marco de torneo (round-robin con roles alternados).
2. Tres agentes: aleatorio, MCTS (UCT), alpha-beta con heurística.
3. Ejecución del torneo en Hex 5×5.
4. Resultados como tabla y gráfica de barras.
5. Ejercicio: mejorar el agente MCTS (reutilización de árbol, rollout guiado).

**Relación con las notas de clase:** sección 05 (MCTS en acción).

**Prerequisitos:** Notebooks 01-03.

---

In [ ]:
# Solo en Colab
# !pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random
import copy
from collections import deque

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB",
    "red": "#E94F37",
    "green": "#27AE60",
    "gray": "#7F8C8D",
    "orange": "#F39C12",
    "purple": "#8E44AD",
}

random.seed(42)
np.random.seed(42)

---

## 1. Clase `Hex` y `MCTSNode` (autocontenido)

In [ ]:
class Hex:
    """Juego de Hex en un tablero de size x size."""

    NEIGHBORS = [(-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0)]

    def __init__(self, size=7):
        self.size = size
        self.board = [[0] * size for _ in range(size)]
        self.current_player = 1

    def actions(self):
        return [(r, c) for r in range(self.size)
                for c in range(self.size) if self.board[r][c] == 0]

    def result(self, action):
        new = copy.deepcopy(self)
        r, c = action
        new.board[r][c] = self.current_player
        new.current_player = 3 - self.current_player
        return new

    def is_terminal(self):
        return self._has_path(1) or self._has_path(2)

    def utility(self, player):
        if self._has_path(player):
            return 1
        if self._has_path(3 - player):
            return -1
        return 0

    def _has_path(self, player):
        n = self.size
        if player == 1:
            start = [(0, c) for c in range(n) if self.board[0][c] == 1]
            goal = lambda r, c: r == n - 1
        else:
            start = [(r, 0) for r in range(n) if self.board[r][0] == 2]
            goal = lambda r, c: c == n - 1
        visited = set()
        queue = deque(start)
        while queue:
            r, c = queue.popleft()
            if (r, c) in visited:
                continue
            visited.add((r, c))
            if goal(r, c):
                return True
            for dr, dc in self.NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and self.board[nr][nc] == player:
                    queue.append((nr, nc))
        return False

---

## 2. Los tres agentes

### 2.1 Agente aleatorio

Elige uniformemente al azar entre las acciones legales.

In [ ]:
def agente_aleatorio(state, player):
    """Elige una acción legal al azar."""
    return random.choice(state.actions())

### 2.2 Agente MCTS (UCT)

Usa Monte Carlo Tree Search con la fórmula UCT para selección.

In [ ]:
class MCTSNode:
    """Nodo del árbol de MCTS."""

    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        self.children = {}
        self.N = 0
        self.Q = 0.0
        self.unexpanded = list(state.actions()) if not state.is_terminal() else []


def uct_value(child, parent_visits, c):
    """Fórmula UCT: explotación + exploración."""
    if child.N == 0:
        return float('inf')
    return child.Q / child.N + c * math.sqrt(math.log(parent_visits) / child.N)


def mcts_uct(game_state, iterations, player, c=1.41):
    """MCTS con selección UCT. Retorna (mejor_acción, raíz)."""
    root = MCTSNode(game_state)

    for _ in range(iterations):
        node = root

        # [M1] Selección con UCT
        while not node.unexpanded and node.children:
            node = max(node.children.values(),
                       key=lambda ch: uct_value(ch, node.N, c))

        # [M2] Expansión
        if node.unexpanded:
            action = node.unexpanded.pop()
            child_state = node.state.result(action)
            child = MCTSNode(child_state, parent=node)
            node.children[action] = child
            node = child

        # [M3] Simulación
        sim = copy.deepcopy(node.state)
        while not sim.is_terminal():
            sim = sim.result(random.choice(sim.actions()))
        reward = sim.utility(player)

        # [M4] Retropropagación
        while node is not None:
            node.N += 1
            node.Q += reward
            node = node.parent

    best_action = max(root.children, key=lambda a: root.children[a].N)
    return best_action, root


def agente_mcts(state, player, iterations=500, c=1.41):
    """Agente MCTS con UCT."""
    action, _ = mcts_uct(state, iterations, player, c)
    return action

### 2.3 Agente Alpha-Beta con heurística

Implementamos alpha-beta con profundidad limitada y una función de evaluación
heurística basada en la **distancia mínima al borde** (BFS). La idea: una
posición es mejor si el jugador está más cerca de conectar sus lados.

In [ ]:
def heuristica_hex(state, player):
    """Función de evaluación heurística para Hex.

    Calcula la diferencia entre la distancia mínima del oponente
    y la del jugador a completar su conexión. Valores positivos
    favorecen a `player`.

    Usa BFS sobre celdas vacías y propias para medir la distancia
    más corta entre los dos lados.
    """
    def distancia_a_conectar(state, p):
        """BFS: distancia mínima para que p conecte sus lados.
        Celdas propias tienen costo 0, vacías costo 1."""
        n = state.size
        INF = float('inf')
        dist = [[INF] * n for _ in range(n)]

        # Cola de prioridad simple (BFS con 0-1 weights)
        if p == 1:  # Negro: fila 0 -> fila n-1
            queue = deque()
            for c in range(n):
                if state.board[0][c] == p:
                    dist[0][c] = 0
                    queue.appendleft((0, c))  # costo 0
                elif state.board[0][c] == 0:
                    dist[0][c] = 1
                    queue.append((0, c))  # costo 1
            goal_row = n - 1
        else:  # Blanco: col 0 -> col n-1
            queue = deque()
            for r in range(n):
                if state.board[r][0] == p:
                    dist[r][0] = 0
                    queue.appendleft((r, 0))
                elif state.board[r][0] == 0:
                    dist[r][0] = 1
                    queue.append((r, 0))
            goal_row = None  # usaremos columna n-1

        while queue:
            r, c = queue.popleft()
            for dr, dc in Hex.NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and state.board[nr][nc] != (3 - p):
                    # Costo: 0 si es nuestra piedra, 1 si está vacía
                    costo = 0 if state.board[nr][nc] == p else 1
                    nuevo_dist = dist[r][c] + costo
                    if nuevo_dist < dist[nr][nc]:
                        dist[nr][nc] = nuevo_dist
                        if costo == 0:
                            queue.appendleft((nr, nc))
                        else:
                            queue.append((nr, nc))

        # Distancia al lado meta
        if p == 1:
            return min(dist[n-1][c] for c in range(n))
        else:
            return min(dist[r][n-1] for r in range(n))

    d_player = distancia_a_conectar(state, player)
    d_oponente = distancia_a_conectar(state, 3 - player)
    # Normalizar a [-1, 1] aproximadamente
    if d_player == 0:
        return 1.0  # jugador ya ganó
    if d_oponente == 0:
        return -1.0  # oponente ya ganó
    return (d_oponente - d_player) / state.size


def alphabeta(state, depth, alpha, beta, player, maximizing):
    """Alpha-beta con profundidad limitada y heurística."""
    if state.is_terminal():
        return state.utility(player), None
    if depth == 0:
        return heuristica_hex(state, player), None

    actions = state.actions()
    best_action = actions[0]

    if maximizing:
        value = -float('inf')
        for action in actions:
            child = state.result(action)
            child_val, _ = alphabeta(child, depth - 1, alpha, beta, player, False)
            if child_val > value:
                value = child_val
                best_action = action
            alpha = max(alpha, value)
            if beta <= alpha:
                break  # poda beta
        return value, best_action
    else:
        value = float('inf')
        for action in actions:
            child = state.result(action)
            child_val, _ = alphabeta(child, depth - 1, alpha, beta, player, True)
            if child_val < value:
                value = child_val
                best_action = action
            beta = min(beta, value)
            if beta <= alpha:
                break  # poda alpha
        return value, best_action


def agente_alphabeta(state, player, depth=3):
    """Agente alpha-beta con profundidad limitada."""
    maximizing = (state.current_player == player)
    _, action = alphabeta(state, depth, -float('inf'), float('inf'), player, maximizing)
    return action

---

## 3. Marco del torneo

Implementamos un torneo **round-robin**: cada par de agentes juega $N$ partidas,
alternando quién es Negro. Registramos victorias, derrotas y tasas.

In [ ]:
def jugar_partida(size, agente1, agente2):
    """Juega una partida entre agente1 (Negro) y agente2 (Blanco).

    Retorna 1 si Negro gana, 2 si Blanco gana.
    """
    state = Hex(size=size)
    while not state.is_terminal():
        if state.current_player == 1:
            action = agente1(state, player=1)
        else:
            action = agente2(state, player=2)
        state = state.result(action)
    return 1 if state.utility(1) == 1 else 2


def torneo_round_robin(size, agentes, nombres, n_partidas_por_par=10):
    """Torneo round-robin entre agentes.

    Cada par juega n_partidas_por_par partidas (mitad como Negro, mitad como Blanco).
    Retorna diccionario con resultados.
    """
    n = len(agentes)
    # Matriz de victorias: victorias[i][j] = victorias del agente i contra j
    victorias = [[0] * n for _ in range(n)]
    partidas_mitad = n_partidas_por_par // 2

    for i in range(n):
        for j in range(i + 1, n):
            print(f"  {nombres[i]} vs {nombres[j]}...", end=" ", flush=True)

            # Agente i como Negro, agente j como Blanco
            for _ in range(partidas_mitad):
                ganador = jugar_partida(size, agentes[i], agentes[j])
                if ganador == 1:
                    victorias[i][j] += 1
                else:
                    victorias[j][i] += 1

            # Agente j como Negro, agente i como Blanco
            for _ in range(partidas_mitad):
                ganador = jugar_partida(size, agentes[j], agentes[i])
                if ganador == 1:
                    victorias[j][i] += 1
                else:
                    victorias[i][j] += 1

            print(f"{victorias[i][j]}-{victorias[j][i]}")

    return victorias

---

## 4. Ejecutar el torneo

Enfrentamos los tres agentes en Hex 5×5. Usamos 500 iteraciones para MCTS
y profundidad 3 para alpha-beta.

**Nota:** Este torneo tarda varios minutos. Puedes reducir `n_partidas_por_par`
para obtener resultados más rápido (con menor confianza estadística).

In [ ]:
random.seed(42)

# Configuración del torneo
SIZE = 5
MCTS_ITERS = 500
AB_DEPTH = 3
N_PARTIDAS = 10  # partidas por par (5 como Negro, 5 como Blanco)

# Definir agentes
agentes = [
    agente_aleatorio,
    lambda s, p: agente_mcts(s, p, iterations=MCTS_ITERS),
    lambda s, p: agente_alphabeta(s, p, depth=AB_DEPTH),
]
nombres = ["Aleatorio", f"MCTS ({MCTS_ITERS} iter)", f"Alpha-Beta (d={AB_DEPTH})"]

print(f"Torneo Round-Robin en Hex {SIZE}×{SIZE}")
print(f"Partidas por par: {N_PARTIDAS} ({N_PARTIDAS//2} como Negro + {N_PARTIDAS//2} como Blanco)")
print("=" * 60)

victorias = torneo_round_robin(SIZE, agentes, nombres, n_partidas_por_par=N_PARTIDAS)

print("=" * 60)
print("¡Torneo completado!")

---

## 5. Resultados: tabla y gráficas

In [ ]:
n_agentes = len(nombres)

# Calcular totales
victorias_total = [sum(victorias[i][j] for j in range(n_agentes) if j != i)
                   for i in range(n_agentes)]
derrotas_total = [sum(victorias[j][i] for j in range(n_agentes) if j != i)
                  for i in range(n_agentes)]
partidas_total = [victorias_total[i] + derrotas_total[i] for i in range(n_agentes)]
tasas = [victorias_total[i] / partidas_total[i] if partidas_total[i] > 0 else 0
         for i in range(n_agentes)]

# Tabla de resultados detallados
print("\n" + "=" * 70)
print("RESULTADOS DETALLADOS (victorias del agente fila vs agente columna)")
print("=" * 70)

# Encabezado
header = f"{'':20s}"
for nombre in nombres:
    header += f"{nombre:>15s}"
print(header)
print("-" * 70)

for i in range(n_agentes):
    row = f"{nombres[i]:20s}"
    for j in range(n_agentes):
        if i == j:
            row += f"{'—':>15s}"
        else:
            row += f"{victorias[i][j]:>15d}"
    print(row)

# Tabla resumen
print("\n" + "=" * 70)
print("RESUMEN")
print("=" * 70)
print(f"{'Agente':20s} {'Victorias':>10s} {'Derrotas':>10s} {'Tasa':>10s}")
print("-" * 50)
for i in range(n_agentes):
    print(f"{nombres[i]:20s} {victorias_total[i]:>10d} {derrotas_total[i]:>10d} {tasas[i]:>9.1%}")

In [ ]:
# Gráfica: tasa de victorias por agente
agent_colors = [COLORS["gray"], COLORS["blue"], COLORS["red"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Tasa de victorias
bars = ax1.bar(nombres, [t * 100 for t in tasas], color=agent_colors, alpha=0.85)
ax1.set_ylabel("Tasa de victorias (%)")
ax1.set_title(f"Torneo Round-Robin — Hex {SIZE}×{SIZE}")
ax1.set_ylim(0, 105)
for bar, tasa in zip(bars, tasas):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"{tasa*100:.0f}%", ha='center', fontweight='bold')

# Gráfica 2: Victorias y derrotas
x = np.arange(n_agentes)
width = 0.35
bars_v = ax2.bar(x - width/2, victorias_total, width, label="Victorias",
                 color=COLORS["green"], alpha=0.8)
bars_d = ax2.bar(x + width/2, derrotas_total, width, label="Derrotas",
                 color=COLORS["red"], alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(nombres, fontsize=9)
ax2.set_ylabel("Partidas")
ax2.set_title("Victorias vs Derrotas")
ax2.legend()

plt.tight_layout()
plt.show()

---

## 6. Análisis de los resultados

Observa:

1. **MCTS vs Aleatorio**: MCTS debería ganar la gran mayoría. Sin ningún
   conocimiento específico del juego, solo con rollouts y UCT, MCTS juega
   mucho mejor que el azar.

2. **MCTS vs Alpha-Beta**: Este es el enfrentamiento más interesante. Alpha-beta
   tiene conocimiento del dominio (la heurística de distancia) pero profundidad
   limitada. MCTS no tiene conocimiento pero puede "ver" mucho más adelante
   mediante rollouts.

3. **Alpha-Beta vs Aleatorio**: Alpha-beta con heurística también debería ganar
   al aleatorio, pero ¿cuánto?

---

## 7. Resultados cara a cara (head-to-head)

Visualicemos los enfrentamientos directos entre pares de agentes.

In [ ]:
# Gráfica de enfrentamientos directos
enfrentamientos = []
for i in range(n_agentes):
    for j in range(i + 1, n_agentes):
        enfrentamientos.append((i, j, victorias[i][j], victorias[j][i]))

fig, axes = plt.subplots(1, len(enfrentamientos), figsize=(5 * len(enfrentamientos), 4))
if len(enfrentamientos) == 1:
    axes = [axes]

for ax, (i, j, vi, vj) in zip(axes, enfrentamientos):
    total = vi + vj
    ax.barh([0, 1], [vi, vj], color=[agent_colors[i], agent_colors[j]], alpha=0.85)
    ax.set_yticks([0, 1])
    ax.set_yticklabels([nombres[i], nombres[j]], fontsize=9)
    ax.set_xlabel("Victorias")
    ax.set_title(f"{nombres[i]}\nvs\n{nombres[j]}", fontsize=10)
    ax.set_xlim(0, total + 1)
    # Anotar valores
    ax.text(vi + 0.2, 0, str(vi), va='center', fontweight='bold')
    ax.text(vj + 0.2, 1, str(vj), va='center', fontweight='bold')

plt.suptitle(f"Enfrentamientos directos — Hex {SIZE}×{SIZE}", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

---

## 8. Ejercicio: mejorar el agente MCTS

El agente MCTS que usamos en el torneo es la versión básica. Hay varias
mejoras posibles:

### Ideas para mejorar:

1. **Reutilización del árbol**: después de cada movimiento, el subárbol
   correspondiente ya tiene estadísticas. En vez de empezar de cero,
   reutiliza ese subárbol como nueva raíz.

2. **Rollout guiado**: en vez de movimientos uniformes, usa una heurística
   simple (e.g., preferir celdas cercanas al centro o a las piedras propias).

3. **Más iteraciones**: simplemente aumentar el presupuesto.

4. **Ajustar $c$**: experimenta con valores distintos de la constante
   de exploración.

**Tu tarea:** implementa al menos una mejora y repite el torneo.
¿Mejora la tasa de victorias?

In [ ]:
# === EJEMPLO: rollout guiado (preferir centro) ===

def rollout_guiado(state):
    """Rollout con preferencia por celdas centrales.

    En vez de elegir uniformemente, da mayor peso a las celdas
    cercanas al centro del tablero.
    """
    sim = copy.deepcopy(state)
    centro = sim.size / 2.0
    while not sim.is_terminal():
        acciones = sim.actions()
        # Pesos inversamente proporcionales a la distancia al centro
        pesos = []
        for r, c in acciones:
            dist = abs(r - centro) + abs(c - centro)
            pesos.append(1.0 / (1.0 + dist))
        # Normalizar
        total = sum(pesos)
        pesos = [w / total for w in pesos]
        # Elegir con probabilidad proporcional al peso
        idx = random.choices(range(len(acciones)), weights=pesos, k=1)[0]
        sim = sim.result(acciones[idx])
    return sim


def mcts_uct_guiado(game_state, iterations, player, c=1.41):
    """MCTS con UCT y rollout guiado."""
    root = MCTSNode(game_state)

    for _ in range(iterations):
        node = root

        while not node.unexpanded and node.children:
            node = max(node.children.values(),
                       key=lambda ch: uct_value(ch, node.N, c))

        if node.unexpanded:
            action = node.unexpanded.pop()
            child_state = node.state.result(action)
            child = MCTSNode(child_state, parent=node)
            node.children[action] = child
            node = child

        # Rollout GUIADO en vez de aleatorio
        sim_final = rollout_guiado(node.state)
        reward = sim_final.utility(player)

        while node is not None:
            node.N += 1
            node.Q += reward
            node = node.parent

    best_action = max(root.children, key=lambda a: root.children[a].N)
    return best_action, root


def agente_mcts_mejorado(state, player, iterations=500, c=1.41):
    """Agente MCTS con rollout guiado."""
    action, _ = mcts_uct_guiado(state, iterations, player, c)
    return action


# === Prueba rápida: MCTS mejorado vs MCTS básico ===
random.seed(42)
n_test = 10
victorias_mejorado = 0

print(f"MCTS mejorado (rollout guiado) vs MCTS básico — {n_test} partidas en Hex 5×5:\n")

for i in range(n_test):
    # Alternar roles
    if i % 2 == 0:
        ganador = jugar_partida(
            size=5,
            agente1=lambda s, p: agente_mcts_mejorado(s, p, 300),
            agente2=lambda s, p: agente_mcts(s, p, 300)
        )
        if ganador == 1:
            victorias_mejorado += 1
    else:
        ganador = jugar_partida(
            size=5,
            agente1=lambda s, p: agente_mcts(s, p, 300),
            agente2=lambda s, p: agente_mcts_mejorado(s, p, 300)
        )
        if ganador == 2:
            victorias_mejorado += 1
    print(f"  Partida {i+1}/{n_test} completada")

print(f"\nMCTS mejorado gana: {victorias_mejorado}/{n_test}")
print(f"MCTS básico gana: {n_test - victorias_mejorado}/{n_test}")
print("\n¿El rollout guiado ayudó? Experimenta con otras mejoras.")

---

## 9. Reflexión final

En este módulo construimos MCTS desde cero, paso a paso:

1. **Rollouts** (notebook 01): evaluar posiciones jugando al azar — estimador Monte Carlo.
2. **MCTS vanilla** (notebook 02): reutilizar información construyendo un árbol.
3. **UCT** (notebook 03): balancear exploración y explotación con la fórmula de bandidos.
4. **Torneo** (notebook 04): verificar que funciona contra otros agentes.

MCTS une tres módulos del curso:
- **Módulo 12** (Monte Carlo): los rollouts son estimadores MC.
- **Módulo 15** (Búsqueda adversarial): la estructura de árbol con propagación de valores.
- **Módulo 17** (Bandidos multibrazo): UCT es UCB1 en cada nodo.

No es un algoritmo nuevo — es la síntesis natural de herramientas que ya teníamos.

---